In [1]:
# Cell 1
!pip install -q scikit-learn joblib numpy


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Cell 2
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression

RNG = np.random.default_rng(42)
N = 4000

In [3]:
# Cell 3 - synthetic dataset
# Ground truth: higher GPA -> higher eligibility; lower income-per-capita
# (more need) -> higher eligibility. documents_complete is deliberately
# NOT a feature - that's a hard business rule in the policy layer
# (see DECISIONS.md), never something the model should learn a weight for.

def make_synthetic_dataset():
    gpa = RNG.uniform(0.0, 5.0, N)
    household_size = RNG.integers(1, 8, N)
    income = RNG.gamma(shape=2.0, scale=1500.0, size=N)
    income_per_capita = income / household_size

    gpa_norm = gpa / 5.0
    log_income_pc = np.log1p(income_per_capita)

    logit = 4.0 * (gpa_norm - 0.5) - 0.6 * (log_income_pc - 7.0) + RNG.normal(0, 0.4, N)
    prob = 1 / (1 + np.exp(-logit))
    label = RNG.binomial(1, prob)

    X = np.column_stack([gpa_norm, log_income_pc, household_size.astype(float)])
    return X, label

X, y = make_synthetic_dataset()
X.shape, y.mean()

((4000, 3), 0.55625)

In [4]:
# Cell 4 - train
model = LogisticRegression(max_iter=1000)
model.fit(X, y)
print('train accuracy:', model.score(X, y))

train accuracy: 0.719


In [5]:
# Cell 5 - sanity check BEFORE saving
# Same claim test_directional_gpa_increases_eligibility checks later -
# verify it here so a broken model never even reaches the repo.

low_gpa_features = [[1.0/5.0, np.log1p(1000.0), 4.0]]
high_gpa_features = [[4.5/5.0, np.log1p(1000.0), 4.0]]
low = model.predict_proba(low_gpa_features)[0][1]
high = model.predict_proba(high_gpa_features)[0][1]
print(f'low-GPA score: {low:.4f}, high-GPA score: {high:.4f}')
assert high >= low, 'model is directionally WRONG on GPA - do not save this artefact'
print('OK: higher GPA -> higher score')

low-GPA score: 0.2649, high-GPA score: 0.8228
OK: higher GPA -> higher score


In [6]:
# Cell 6 - save
bundle = {
    'sklearn_model': model,
    'feature_order': ['gpa_normalised', 'log_income_per_capita', 'household_size'],
    'model_version': 'rasheed-lr-v1',
}
joblib.dump(bundle, 'rasheed_lr_v1.joblib')
print('Saved rasheed_lr_v1.joblib - move this file to models/rasheed_lr_v1.joblib in your repo')

Saved rasheed_lr_v1.joblib - move this file to models/rasheed_lr_v1.joblib in your repo
